# MetaCog-Triage v2 Run (Kaggle)

**Before running:** upload the `metacog_triage_release` folder as a Kaggle *Dataset* (after running `prepare_release.py` locally so `tasks/` is populated), attach it to this notebook, and enable a GPU (T4 is enough).

Set `PKG_INPUT` below to the attached dataset path.

In [1]:
from pathlib import Path
import os, shutil

PKG_INPUT = Path("/kaggle/input/datasets/siddhantdamre11/metacog-triage-release/metacog_triage_release")
WORK_DIR = Path("/kaggle/working/pkg")

if not PKG_INPUT.exists():
    print("Exact path not found. Searching for prepare_release.py...")
    matches = list(Path("/kaggle/input").rglob("prepare_release.py"))
    print("Matches:", matches)
    if not matches:
        raise FileNotFoundError("Could not find prepare_release.py under /kaggle/input")
    PKG_INPUT = matches[0].parent

print("Using package input:", PKG_INPUT)

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

shutil.copytree(PKG_INPUT, WORK_DIR)

os.chdir(WORK_DIR)
print("Current directory:", os.getcwd())
print("Top-level files:")
print(os.listdir("."))

Using package input: /kaggle/input/datasets/siddhantdamre11/metacog-triage-release/metacog_triage_release
Current directory: /kaggle/working/pkg
Top-level files:
['METACOG_TRIAGE_COORDINATOR_REPORT.md', 'LICENSE', 'results', 'LOCAL_EXECUTION_CHECKLIST.md', 'RELEASE_CHECKLIST.md', 'kaggle_run_v2.ipynb', 'README.md', 'requirements.txt', 'prepare_release.py', 'analysis', 'CITATION.cff', 'RELEASE_READINESS_REPORT.md', 'tasks', 'src', 'baselines', 'AUDIT_REPORT.md', 'EXTERNAL_REVIEW_PLAN.md', 'paper', 'MACHINE_VERIFICATION_REPORT.md']


In [6]:
import torch, transformers, accelerate, sentencepiece

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("transformers:", transformers.__version__)

torch: 2.10.0+cu128
cuda available: True
device: Tesla T4
transformers: 5.0.0


In [7]:
!python prepare_release.py

Repo root: /kaggle/working
Copying frozen artifacts...
  MISSING (skipped): benchmarks/exec_meta_adapt/frontier/frontier_tasks_metacog.jsonl
  MISSING (skipped): results/frontier_local/full_40/comparison_table.md
  MISSING (skipped): results/frontier_local/full_40/qwen_results.json
  MISSING (skipped): results/frontier_local/full_40/smollm_results.json
  MISSING (skipped): results/frontier_local/open_model_expansion/full_40_single/comparison_table.md
  MISSING (skipped): submission/frontier_local_metacognition/frontier_local_metacognition_behavior.svg

0/6 required artifacts copied.

Validating v1 task set...
OK: 40 tasks, all schema checks passed
  by type:    {'hidden_state_uncertainty': 10, 'adversarial_trust': 10, 'overflow_mismatch': 10, 'clear_commit': 10}
  by gold:    {'ABSTAIN': 10, 'ESCALATE': 20, 'COMMIT': 10}
  by variant: {'standard': 40}
  by split:   {'train': 28, 'heldout': 12}
  by domain:  {'benchmark': 13, 'cyber': 13, 'clinical': 14}

Generating v2 task set...
Wrote

In [8]:
!python baselines/baselines.py --tasks tasks/metacog_tasks_v1.jsonl --output analysis/baseline_results_v1_kaggle.csv
!python baselines/baselines.py --tasks tasks/metacog_tasks_v2.jsonl --output analysis/baseline_results_v2_kaggle.csv

40 tasks; gold distribution {'ABSTAIN': 10, 'ESCALATE': 20, 'COMMIT': 10}; majority class ESCALATE
{'baseline': 'always_commit', 'n': 40, 'accuracy': 0.25, 'acc_gold_COMMIT': 1.0, 'acc_gold_ABSTAIN': 0.0, 'acc_gold_ESCALATE': 0.0, 'acc_standard': 0.25, 'acc_control': '', 'over_escalation': 0.0, 'over_abstention': 0.0, 'bluff': 0.75}
{'baseline': 'always_abstain', 'n': 40, 'accuracy': 0.25, 'acc_gold_COMMIT': 0.0, 'acc_gold_ABSTAIN': 1.0, 'acc_gold_ESCALATE': 0.0, 'acc_standard': 0.25, 'acc_control': '', 'over_escalation': 0.0, 'over_abstention': 0.5, 'bluff': 0.0}
{'baseline': 'always_escalate', 'n': 40, 'accuracy': 0.5, 'acc_gold_COMMIT': 0.0, 'acc_gold_ABSTAIN': 0.0, 'acc_gold_ESCALATE': 1.0, 'acc_standard': 0.5, 'acc_control': '', 'over_escalation': 0.25, 'over_abstention': 0.0, 'bluff': 0.0}
{'baseline': 'majority_class', 'n': 40, 'accuracy': 0.5, 'acc_gold_COMMIT': 0.0, 'acc_gold_ABSTAIN': 0.0, 'acc_gold_ESCALATE': 1.0, 'acc_standard': 0.5, 'acc_control': '', 'over_escalation': 0.

In [9]:
!python baselines/recompute_calibration.py \
  results/v1_four_model/qwen_results.json \
  results/v1_four_model/smollm_results.json \
  results/v1_four_model/granite_results.json \
  results/v1_four_model/tinyllama_results.json


=== qwen (qwen_results.json) ===
n=40 parsed=40 acc_all=0.7250 acc_parsed=0.7250
mean_conf=0.1913 conf_correct=0.2638 conf_wrong=0.0000 discrimination_gap=+0.2638
calibration_gap=-0.5337 ECE=0.5338 Brier=0.5262 confident_error_rate=0.0000
  bucket [0.0,0.2]: n=32 acc=0.6562 conf=0.0000
  bucket [0.8,1.0]: n=8 acc=1.0000 conf=0.9563
  gold COMMIT: n=10 acc=0.8000 conf(parsed)=0.7650
  gold ABSTAIN: n=10 acc=0.1000 conf(parsed)=0.0000
  gold ESCALATE: n=20 acc=1.0000 conf(parsed)=0.0000

=== smollm (smollm_results.json) ===
n=40 parsed=40 acc_all=0.7500 acc_parsed=0.7500
mean_conf=0.8912 conf_correct=0.9183 conf_wrong=0.8100 discrimination_gap=+0.1083
calibration_gap=+0.1412 ECE=0.1413 Brier=0.1723 confident_error_rate=0.1500
  bucket [0.6,0.8]: n=4 acc=0.0000 conf=0.7000
  bucket [0.8,1.0]: n=36 acc=0.8333 conf=0.9125
  gold COMMIT: n=10 acc=1.0000 conf(parsed)=0.9550
  gold ABSTAIN: n=10 acc=0.0000 conf(parsed)=0.8100
  gold ESCALATE: n=20 acc=1.0000 conf(parsed)=0.9000

=== granite (

In [10]:
# Generate + validate v2 tasks if not already present in the dataset
import os, subprocess, sys
if not os.path.exists('tasks/metacog_tasks_v2.jsonl'):
    subprocess.run([sys.executable, 'tasks/generate_tasks_v2.py'], check=True)
subprocess.run([sys.executable, 'tasks/validate_tasks.py', 'tasks/metacog_tasks_v2.jsonl', '--expect-v2'], check=True)

OK: 200 tasks, all schema checks passed
  by type:    {'adversarial_trust': 50, 'clear_commit': 50, 'hidden_state_uncertainty': 50, 'overflow_mismatch': 50}
  by gold:    {'ESCALATE': 80, 'COMMIT': 70, 'ABSTAIN': 50}
  by variant: {'standard': 120, 'control': 80}
  by split:   {'train': 140, 'heldout': 60}
  by domain:  {'clinical': 68, 'benchmark': 68, 'cyber': 64}
OK: v2 balance and decoupling checks passed


CompletedProcess(args=['/usr/bin/python3', 'tasks/validate_tasks.py', 'tasks/metacog_tasks_v2.jsonl', '--expect-v2'], returncode=0)

In [11]:
# Baselines on v2 (fast, CPU)
import subprocess, sys
subprocess.run([sys.executable, 'baselines/baselines.py',
                '--tasks', 'tasks/metacog_tasks_v2.jsonl',
                '--output', 'analysis/baseline_results_v2.csv'], check=True)

200 tasks; gold distribution {'ESCALATE': 80, 'COMMIT': 70, 'ABSTAIN': 50}; majority class ESCALATE
{'baseline': 'always_commit', 'n': 200, 'accuracy': 0.35, 'acc_gold_COMMIT': 1.0, 'acc_gold_ABSTAIN': 0.0, 'acc_gold_ESCALATE': 0.0, 'acc_standard': 0.25, 'acc_control': 0.5, 'over_escalation': 0.0, 'over_abstention': 0.0, 'bluff': 0.65}
{'baseline': 'always_abstain', 'n': 200, 'accuracy': 0.25, 'acc_gold_COMMIT': 0.0, 'acc_gold_ABSTAIN': 1.0, 'acc_gold_ESCALATE': 0.0, 'acc_standard': 0.25, 'acc_control': 0.25, 'over_escalation': 0.0, 'over_abstention': 0.4, 'bluff': 0.0}
{'baseline': 'always_escalate', 'n': 200, 'accuracy': 0.4, 'acc_gold_COMMIT': 0.0, 'acc_gold_ABSTAIN': 0.0, 'acc_gold_ESCALATE': 1.0, 'acc_standard': 0.5, 'acc_control': 0.25, 'over_escalation': 0.25, 'over_abstention': 0.0, 'bluff': 0.0}
{'baseline': 'majority_class', 'n': 200, 'accuracy': 0.4, 'acc_gold_COMMIT': 0.0, 'acc_gold_ABSTAIN': 0.0, 'acc_gold_ESCALATE': 1.0, 'acc_standard': 0.5, 'acc_control': 0.25, 'over_esc

CompletedProcess(args=['/usr/bin/python3', 'baselines/baselines.py', '--tasks', 'tasks/metacog_tasks_v2.jsonl', '--output', 'analysis/baseline_results_v2.csv'], returncode=0)

In [ ]:
MODELS = ["qwen", "smollm", "granite", "tinyllama"]

import subprocess, sys, os

os.makedirs("results/v2_run1", exist_ok=True)

subprocess.run([
    sys.executable,
    "src/run_benchmark.py",
    "--models", *MODELS,
    "--tasks", "tasks/metacog_tasks_v2.jsonl",
    "--output", "results/v2_run1/"
], check=True)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 375.32it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[qwen] 20/200 tasks done
[qwen] 40/200 tasks done
[qwen] 60/200 tasks done
[qwen] 80/200 tasks done
[qwen] 100/200 tasks done
[qwen] 120/200 tasks done
[qwen] 140/200 tasks done
[qwen] 160/200 tasks done
[qwen] 180/200 tasks done
[qwen] 200/200 tasks done


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 227.28it/s, Materializing param=model.norm.weight]                              


[smollm] 20/200 tasks done
[smollm] 40/200 tasks done
[smollm] 60/200 tasks done
[smollm] 80/200 tasks done
[smollm] 100/200 tasks done
[smollm] 120/200 tasks done
[smollm] 140/200 tasks done
[smollm] 160/200 tasks done
[smollm] 180/200 tasks done
[smollm] 200/200 tasks done


Loading weights: 100%|██████████| 362/362 [00:01<00:00, 253.31it/s, Materializing param=model.norm.weight]                              


In [12]:
# Main run: ~1-2h for 4 models x 200 tasks on a T4
import subprocess, sys
subprocess.run([sys.executable, 'src/run_benchmark.py',
                '--models', *MODELS,
                '--tasks', 'tasks/metacog_tasks_v2.jsonl',
                '--output', 'results/v2_run1/'], check=True)

NameError: name 'MODELS' is not defined

In [ ]:
# Calibration recompute on the new v2 results
import subprocess, sys, glob
subprocess.run([sys.executable, 'baselines/recompute_calibration.py',
                *sorted(glob.glob('results/v2_run1/*_results.json'))], check=True)

In [ ]:
# Bundle outputs for download (Kaggle 'Output' tab)
import shutil
shutil.make_archive('/kaggle/working/v2_results_bundle', 'zip', 'results/v2_run1')
shutil.copy('analysis/baseline_results_v2.csv', '/kaggle/working/')
print('Done. Download v2_results_bundle.zip and baseline_results_v2.csv from the notebook Output tab.')

## After downloading
1. Copy `v2_run1/` into `metacog_triage_release/results/` locally.
2. Check the **By Variant** section of `comparison_table.md` — standard-vs-control gap is the headline number.
3. Compare baselines against the pre-registered predictions in `analysis/baseline_summary.md`.
4. Update paper §5 and re-run the push script.